# Bangla Dataset Preperation for Digital Forensic Analysis From Social Media Data

* **Bangla Profane Words Datasets:**  Nirmol Dataset, ToxLex Dataset
* **(Bangla Transliteration Dataset:** BANTH Dataset
* **Bangla Forensic Dataset:** BDSHS, BanHate Dataset

## Push Data to github Repository

In [21]:
!pip install python-dotenv
import os
import subprocess
from pathlib import Path
from dotenv import load_dotenv
import os

def upload_to_github(file_path, commit_message, branch="development"):
    # --- CONFIGURATION ---
    # Load .env file
    load_dotenv()
    # Access variables
    GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
    GITHUB_USER = os.getenv("GITHUB_USER")
    REPO_NAME = os.getenv("REPO_NAME")
    EMAIL = os.getenv("EMAIL")

    # Authenticated URL
    repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git"

    # --- DIRECTORY MANAGEMENT ---
    repo_path = Path(REPO_NAME)

    if not repo_path.exists():
        subprocess.run(["git", "clone", repo_url], check=True)

    # Change to repo directory
    original_dir = os.getcwd()
    os.chdir(repo_path)

    try:
        # Fetch all branches
        subprocess.run(["git", "fetch", "--all"], check=True)

        # Check if developer branch exists
        result = subprocess.run(["git", "show-ref", "--verify", f"refs/heads/{branch}"],
                              capture_output=True, text=True)

        if result.returncode == 0:
            # Branch exists locally, switch to it
            subprocess.run(["git", "checkout", branch], check=True)
            subprocess.run(["git", "pull", "origin", branch], check=True)
        else:
            # Check if branch exists remotely
            result = subprocess.run(["git", "ls-remote", "--heads", "origin", branch],
                                  capture_output=True, text=True)
            if result.stdout.strip():
                # Branch exists remotely, checkout and track
                subprocess.run(["git", "checkout", "-b", branch, f"origin/{branch}"], check=True)
            else:
                # Create new branch
                subprocess.run(["git", "checkout", "-b", branch], check=True)

        # Copy the file
        filename = os.path.basename(file_path)
        subprocess.run(["cp", file_path, "."], check=True)

        # Configure git
        subprocess.run(["git", "config", "--global", "user.email", EMAIL], check=True)
        subprocess.run(["git", "config", "--global", "user.name", GITHUB_USER], check=True)

        # Add, commit, and push to developer branch
        subprocess.run(["git", "add", filename], check=True)
        subprocess.run(["git", "commit", "-m", commit_message], check=True)
        subprocess.run(["git", "push", "origin", branch], check=True)

        print(f"Successfully pushed {filename} to {REPO_NAME}/{branch} branch")

    except subprocess.CalledProcessError as e:
        print(f"Error during git operation: {e}")
        raise
    finally:
        # Return to original directory
        os.chdir(original_dir)

base_dir = "/home/sakib/Documents/"

## Download CSV/Excel Files from Link and Process Data 

In [ ]:
import pandas as pd
import requests
from io import BytesIO

def load_file(url):
    try:
        print("Downloading file...")
        response = requests.get(url, allow_redirects=True)
        response.raise_for_status()  # Raise error if request fails

        # Check Content-Type from HTTP header
        content_type = response.headers.get('Content-Type', '').lower()

        if 'excel' in content_type or 'spreadsheet' in content_type:
            df = pd.read_excel(BytesIO(response.content))
        elif 'csv' in content_type or 'text' in content_type:
            df = pd.read_csv(BytesIO(response.content))
        else:
            # Fallback: try Excel first, then CSV
            try:
                df = pd.read_excel(BytesIO(response.content))
            except:
                df = pd.read_csv(BytesIO(response.content))
        return df

    except Exception as e:
        print(f"❌ Error loading file: {e}")
        return None

# Bangla Profane Words Datasets
* **Input:**  Nirmol Dataset, ToxLex Dataset
* **Output:** Profane Pairs csv file

### Nirmol Dataset

In [12]:
#Clone Nirmol Repository
!git clone https://github.com/Sigmakib2/Nirmol.git

fatal: destination path 'Nirmol' already exists and is not an empty directory.


In [13]:
nirmol_data = pd.read_csv("Nirmol/datasets/Nirmol-v1-dataset.csv", index_col=False)
# Rename first column to standard name
nirmol_data.rename(columns={nirmol_data.columns[0]: "input_data"}, inplace=True)
nirmol_data = nirmol_data[['input_data']]
print(f"Nirmol shape: {nirmol_data.shape}")
nirmol_data.head()

Nirmol shape: (1181, 1)


,input_data
0,চোদানীর পোলা
1,মাংগের নাতী
2,হোগার নাতী
3,আবাল জাতের মায়েরে চুদি
4,বেইসসার পোলা


### ToxLex Dataset

In [15]:
toxlex_data = load_file(
    "https://data.mendeley.com/public-files/datasets/9pz8ssmc49/files/"
    "c13b54f5-30d7-4357-8c49-37bc46e510ab/file_downloaded"
)

toxlex_data = toxlex_data.rename(columns={'Base_bigram': 'input_data'})
toxlex_data = toxlex_data[['input_data']]
print(f"ToxLex shape: {toxlex_data.shape}")
toxlex_data.head()

ToxLex shape: (1959, 1)


,input_data
0,অক্ষম পুরুষের
1,অডিও সেক্স
2,অনৈসলামিক কাজ
3,অন্ডকোষ কাটলেই
4,অন্ডকোষ হীন


### Merge Profane Words Dataset

In [17]:
profane_words = (
    pd.concat([toxlex_data, nirmol_data], ignore_index=True)
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)
print(f"✅ Merged profane word list: {profane_words.shape[0]} unique entries")
profane_words.head()

✅ Merged profane word list: 3117 unique entries


,input_data
0,অক্ষম পুরুষের
1,অডিও সেক্স
2,অনৈসলামিক কাজ
3,অন্ডকোষ কাটলেই
4,অন্ডকোষ হীন


### Generate Masked Profane Pairs

In [18]:
import pandas as pd
import random
import torch
from torch.utils.data import Dataset, DataLoader

def generate_masked_pairs_with_dataloader(profane_words_df, column_name="input_data", batch_size=32, shuffle=True):
    """
    Generate masked pairs and also return DataLoader ready for training.

    Returns:
    - DataFrame with masked pairs
    - DataLoader for training
    - char2idx mapping
    - idx2char mapping
    - max_len
    """

    def mask_word(w: str):
        """Generate various masked versions of a word."""
        styles = []

        if len(w) > 2:
            pos = random.randint(1, len(w)-2)
            tmp = w[:pos] + "*" + w[pos+1:]
            styles.append(tmp)

        styles.append("-".join(list(w)))
        styles.append("/".join(list(w)))

        n_mask = random.randint(1, max(1, len(w)//2))
        tmp = list(w)
        mask_pos = random.sample(range(len(w)), min(n_mask, len(w)))
        for p in mask_pos:
            tmp[p] = "*"
        styles.append("".join(tmp))

        return styles

    # Debug: Check if column exists
    if column_name not in profane_words_df.columns:
        print(f"Error: Column '{column_name}' not found in DataFrame")
        print(f"Available columns: {list(profane_words_df.columns)}")
        return pd.DataFrame(), None, {}, {}, 0

    # Debug: Check if column has any data
    print(f"Found {len(profane_words_df)} rows in DataFrame")
    print(f"Sample values from '{column_name}': {profane_words_df[column_name].head(10).tolist()}")

    # Generate pairs
    pairs = []
    for idx, w in enumerate(profane_words_df[column_name]):
        try:
            # Skip if not a string or empty
            if not isinstance(w, str) or len(w.strip()) == 0:
                print(f"Skipping row {idx}: empty or non-string value: {w}")
                continue

            w = w.strip()  # Clean whitespace
            masked_versions = mask_word(w)
            for m in masked_versions:
                pairs.append((m, w))
        except Exception as e:
            print(f"Error processing word '{w}' at row {idx}: {e}")
            continue

    # Check if pairs were generated
    if len(pairs) == 0:
        print("ERROR: No training pairs were generated!")
        print("Please check that your DataFrame contains valid string values in the specified column.")
        return pd.DataFrame(), None, {}, {}, 0

    print(f"Training pairs generated: {len(pairs)}")

    # Create character mappings
    all_chars = set("".join([p[0] for p in pairs])) | set("".join([p[1] for p in pairs]))
    all_chars = sorted(list(all_chars))
    char2idx = {c: i+1 for i, c in enumerate(all_chars)}
    char2idx["<pad>"] = 0
    idx2char = {i: c for c, i in char2idx.items()}

    # Calculate max length
    max_len = max(max(len(a), len(b)) for a, b in pairs)
    print(f"Max sequence length: {max_len}")
    print(f"Vocabulary size: {len(char2idx)}")

    # Encoding functions
    def encode(s: str) -> torch.Tensor:
        arr = [char2idx.get(c, 0) for c in s]
        arr += [0] * (max_len - len(arr))
        return torch.tensor(arr, dtype=torch.long)

    def decode(arr) -> str:
        if torch.is_tensor(arr):
            arr = arr.tolist()
        return "".join(idx2char[int(i)] for i in arr if i != 0)

    # Create Dataset class
    class MaskDataset(Dataset):
        def __init__(self, pairs_list):
            self.X = [encode(m) for m, _ in pairs_list]
            self.Y = [encode(w) for _, w in pairs_list]

        def __len__(self):
            return len(self.X)

        def __getitem__(self, i):
            return self.X[i], self.Y[i]

    # Create Dataset and DataLoader
    dataset = MaskDataset(pairs)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

    # Create dataframe
    profane_pairs = pd.DataFrame(pairs, columns=["masked_word", "original_word"])
    profane_pairs.rename(columns={"masked_word": "input_data", "original_word": "target_data"}, inplace=True)
    return profane_pairs, dataloader, char2idx, idx2char, max_len


# Let's first check what's in your DataFrame
print("First, let's examine your DataFrame:")
print(f"Shape: {profane_words.shape}")
print(f"Columns: {profane_words.columns.tolist()}")
print("\nFirst few rows:")
print(profane_words.head())

# Then call the function
profane_pairs, train_loader, char2idx, idx2char, max_len = generate_masked_pairs_with_dataloader(
    profane_words,
    column_name="input_data",  # Make sure this matches your actual column name
    batch_size=64,
    shuffle=True
)

# Only save if we have data
if len(profane_pairs) > 0:
    profane_pairs.to_csv("profane_pairs.csv", index=False, encoding="utf-8-sig")
    print(f"\nSuccessfully saved {len(profane_pairs)} pairs to profane_pairs.csv")
    print("\nSample pairs:")
    print(profane_pairs.head())
else:
    print("\nNo data to save. Please check your input DataFrame and column name.")


profane_pairs, train_loader, char2idx, idx2char, max_len = generate_masked_pairs_with_dataloader(
    profane_words,
    column_name="input_data",
    batch_size=64,
    shuffle=True
)

# Save the pairs to CSV
profane_pairs.to_csv("profane_pairs.csv", index=False, encoding="utf-8-sig")


First, let's examine your DataFrame:
Shape: (3117, 1)
Columns: ['input_data']

First few rows:
       input_data
0   অক্ষম পুরুষের
1      অডিও সেক্স
2   অনৈসলামিক কাজ
3  অন্ডকোষ কাটলেই
4     অন্ডকোষ হীন
Found 3117 rows in DataFrame
Sample values from 'input_data': ['অক্ষম পুরুষের', 'অডিও সেক্স', 'অনৈসলামিক কাজ', 'অন্ডকোষ কাটলেই', 'অন্ডকোষ হীন', 'অবৈধ জালিম', 'অবৈধ পোলা', 'অবৈধ প্রেম', 'অবৈধ ফসল', 'অবৈধ সন্তান']
Training pairs generated: 12459
Max sequence length: 87
Vocabulary size: 110

Successfully saved 12459 pairs to profane_pairs.csv

Sample pairs:
                  input_data    target_data
0              অক্*ম পুরুষের  অক্ষম পুরুষের
1  অ-ক-্-ষ-ম- -প-ু-র-ু-ষ-ে-র  অক্ষম পুরুষের
2  অ/ক/্/ষ/ম/ /প/ু/র/ু/ষ/ে/র  অক্ষম পুরুষের
3              অ***ম পুরুষের  অক্ষম পুরুষের
4                 অডিও স*ক্স     অডিও সেক্স
Found 3117 rows in DataFrame
Sample values from 'input_data': ['অক্ষম পুরুষের', 'অডিও সেক্স', 'অনৈসলামিক কাজ', 'অন্ডকোষ কাটলেই', 'অন্ডকোষ হীন', 'অবৈধ জালিম', 'অবৈধ পোলা', 'অবৈধ

In [23]:
# Push Data to the Github
upload_to_github(base_dir+"profane_pairs.csv", "pushed profane pair merged dataset", "main")

Switched to branch 'main'


Your branch is up to date with 'origin/main'.


From https://github.com/sakibmuhtadee/MICT_2000_Project
 * branch            main       -> FETCH_HEAD


Already up to date.
[main 9d71e19] pushed profane pair merged dataset
 1 file changed, 12460 insertions(+)
 create mode 100644 profane_pairs.csv
Successfully pushed profane_pairs.csv to MICT_2000_Project/main branch


To https://github.com/sakibmuhtadee/MICT_2000_Project.git
   93cf5c7..9d71e19  main -> main


# Bangla Transliteration Dataset

## Load BANTH Dataset 

In [24]:
import pandas as pd
from datasets import load_dataset

# Load BanTH dataset
print("Loading BanTH dataset from Hugging Face...")
dataset = load_dataset("aplycaebous/BanTH")

# Convert to DataFrames
train_df = dataset['train'].to_pandas()
test_df = dataset['test'].to_pandas()

# Check column names first
print("Available columns:", train_df.columns.tolist())

# Rename columns
train_df = train_df.rename(columns={'Text': 'input_data', 'bangla': 'target_data'})
test_df = test_df.rename(columns={'Text': 'input_data', 'bangla': 'target_data'})

# Keep only input and target columns (using double brackets)
train_df = train_df[['input_data', 'target_data']]
test_df = test_df[['input_data', 'target_data']]

# Combine datasets
combined_df = pd.concat([train_df, test_df], ignore_index=True)

print(f"\nDataset shapes:")
print(f"  Train: {train_df.shape}")
print(f"  Test: {test_df.shape}")
print(f"  Combined: {combined_df.shape}")

# Save to CSV
train_df.to_csv('banth_train.csv', index=False)
test_df.to_csv('banth_test.csv', index=False)
combined_df.to_csv('transliteration_dataset.csv', index=False)

print("\n✅ Files saved successfully!")
print("\nSample data:")
print(combined_df.head(3))

# Verify the columns
print(f"\nColumns in saved files: {combined_df.columns.tolist()}")

Loading BanTH dataset from Hugging Face...


Available columns: ['Text', 'Label', 'Political', 'Religious', 'Gender', 'Personal Offense', 'Abusive/Violence', 'Origin', 'Body Shaming', 'Misc', 'bangla', 'english']

Dataset shapes:
  Train: (29879, 2)
  Test: (3735, 2)
  Combined: (33614, 2)

✅ Files saved successfully!

Sample data:
                                          input_data  \
0                 Dakha mon tah vorah galoh... ❤❤🎉🎉🎉   
1  BGB er moton ei gulare Chakritheke obbahoti de...   
2  Or 2 hat kata hok  \nAwamiliger ekta kormio je...   

                                         target_data  
0                        দেখা মন টা ভোরা গেল...❤❤🎉🎉🎉  
1       বিজিবি এর মত এই গুলারে চক্রকে বাধা দেয়া হোক  
2  অর ২ হাত কাটা হোক আওয়ামী লীগের একটা কাজিও যাত...  

Columns in saved files: ['input_data', 'target_data']


In [25]:
# Push Data to the Github
upload_to_github(base_dir+"transliteration_dataset.csv", "pushed transliteration dataset", "main")

Already on 'main'


Your branch is up to date with 'origin/main'.


From https://github.com/sakibmuhtadee/MICT_2000_Project
 * branch            main       -> FETCH_HEAD


Already up to date.
[main 353cef0] pushed transliteration dataset
 1 file changed, 36456 insertions(+)
 create mode 100644 transliteration_dataset.csv
Successfully pushed transliteration_dataset.csv to MICT_2000_Project/main branch


To https://github.com/sakibmuhtadee/MICT_2000_Project.git
   9d71e19..353cef0  main -> main


# Bangla Digital Forensic Datasets

## BD-SHS Dataset

In [27]:
!pip install kagglehub
!pip install datasets
!pip install scikit-learn
!pip install pandas

In [28]:
import kagglehub
import pandas as pd
import os
from datasets import load_dataset
from sklearn.model_selection import train_test_split

In [33]:
def load_bdshs():
    print("=" * 55)
    print("  Loading BD-SHS dataset from Kaggle...")
    print("=" * 55)

    path = kagglehub.dataset_download("naurosromim/bdshs")
    print(f"Downloaded to: {path}")

    # Find all CSVs
    csv_files = []
    for root, dirs, files in os.walk(path):
        for file in files:
            if file.endswith(".csv"):
                csv_files.append(os.path.join(root, file))

    if not csv_files:
        raise FileNotFoundError("No CSV files found in BD-SHS download.")

    # Load all CSVs
    dataframes = {}
    for csv_path in csv_files:
        name = os.path.basename(csv_path)
        try:
            df_tmp = pd.read_csv(csv_path, encoding="utf-8")
        except UnicodeDecodeError:
            df_tmp = pd.read_csv(csv_path, encoding="utf-8-sig")

        dataframes[name] = df_tmp
        print(f"  Found: {name}  →  shape: {df_tmp.shape}")
    # Pick largest dataset
    df = max(dataframes.values(), key=len).copy()
    print(f"\n  Selected dataset shape: {df.shape}")

    label_map = {
        "callToViolence": "violence",
        "callToViolence_gender": "violence",
        "callToViolence_religion_slander": "violence",
        "gender": "hate_speech",
        "gender_religion_slander": "hate_speech",
        "religion_slander": "hate_speech",
        "slander": "offensive"
    }
    
    # 1. Replace the specific strings using the map
    # 2. Fill any remaining NaN values (or unmapped values) with 'normal'
    df['type'] = df['type'].map(label_map).fillna("normal")
    df = df.rename(columns={
    'sentence': 'input_data',
    'type': 'target_data'})
    df = df[["input_data","target_data"]]
    return df
bd_shs = load_bdshs()


  Loading BD-SHS dataset from Kaggle...
Downloaded to: /home/sakib/.cache/kagglehub/datasets/naurosromim/bdshs/versions/2
  Found: train.csv  →  shape: (40224, 4)
  Found: val.csv  →  shape: (5028, 4)
  Found: test.csv  →  shape: (5029, 4)

  Selected dataset shape: (40224, 4)


In [34]:
print(bd_shs.shape)

(40224, 2)


In [31]:
bd_shs["target_data"].unique()

array(['violence', 'normal', 'offensive', 'hate_speech'], dtype=object)

## BanHATE Dataset

In [36]:
import pandas as pd
!wget https://huggingface.co/datasets/aplycaebous/BanHate/blob/main/Dataset.json
def prepare_dataset(json_path):
    # Load dataset
    df = pd.read_json(json_path)
    
    # Label mapping rules
    label_map = {
        'violence': ['abusive/violence'],
        'hate_speech': ['religious', 'gender', 'origin', 'political'],
        'cyberbully': ['personal offence', 'body shaming']
    }

    # Function to map labels
    def get_mapped_label(label_str):
        if pd.isna(label_str):
            return 'normal'

        text = str(label_str).lower()

        if any(word in text for word in label_map['violence']):
            return 'violence'

        if any(word in text for word in label_map['hate_speech']):
            return 'hate_speech'

        if any(word in text for word in label_map['cyberbully']):
            return 'cyberbully'

        return 'offensive'

    # Rename column
    df = df.rename(columns={
        'Comment': 'input_data'
    })

    # Create target column
    df['target_data'] = df['Hate Category'].apply(get_mapped_label)

    # Keep only required columns
    df = df[['input_data', 'target_data']]

    return df

ban_hate= prepare_dataset("Dataset.json")

--2026-05-22 11:55:25--  https://huggingface.co/datasets/aplycaebous/BanHate/blob/main/Dataset.json
Resolving huggingface.co (huggingface.co)... 13.35.20.104, 13.35.20.37, 13.35.20.8, ...
Connecting to huggingface.co (huggingface.co)|13.35.20.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 79099 (77K) [text/html]
Saving to: ‘Dataset.json.1’

Dataset.json.1      100%[===================>]  77.25K  --.-KB/s    in 0.009s  

2026-05-22 11:55:25 (8.65 MB/s) - ‘Dataset.json.1’ saved [79099/79099]



In [37]:
ban_hate.head()

,input_data,target_data
0,উপস্থাপিকার চুলকানি বেশি,cyberbully
1,ইভানা ও কোন দিক দিয়ে সুন্দরী ওর কোন চেহারা হইলো,cyberbully
2,ইভানা মাইয়াডার যে চেহারা ও এই চেহারা নিয়ে নাইক...,cyberbully
3,"এগুলা চিনলে কোন ধোন হবে ভাই,,,ভাল কিছু করেন,,,...",violence
4,"ইবানের বিছি গেসে,,😂😂",hate_speech


In [39]:
ban_hate["target_data"].unique()

array(['cyberbully', 'violence', 'hate_speech', 'normal'], dtype=object)

In [43]:
# Combine datasets
combined_df = pd.concat([ban_hate, bd_shs], ignore_index=True)

# Remove duplicate comments
combined_df = combined_df.drop_duplicates(subset=['input_data'])

# Shuffle rows randomly
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Check conflicting labels
duplicates = combined_df.groupby('input_data')['target_data'].nunique()

conflicting = duplicates[duplicates > 1]

print("Conflicting samples:", len(conflicting))

# Preview
print(combined_df.head())

Conflicting samples: 0
                                          input_data target_data
0         পাকিস্তানের একটি বড় গরুর হাট দেখার ইচ্ছে।      normal
1  ফুটবল খেলা পা দিয়ে। হাত সিমানার ভিতরে গেলে কি ...      normal
2  বাংলাদেশ সরকার কে বলছি জেলের ভাত নষ্ট করার চেয...      normal
3               মেয়েটা এতো সাজুগুজু না করলেও পারতেন।      normal
4                           হায়রে দুনিয়া কতরকম মানুষ      normal


In [45]:
print(combined_df.shape)

(59410, 2)


In [46]:
combined_df.to_csv('forensic_dataset_bn.csv', index=False)

In [47]:
# Push Data to the Github
upload_to_github(base_dir+"forensic_dataset_bn.csv", "pushed forensic dataset", "main")

Already on 'main'


Your branch is up to date with 'origin/main'.


From https://github.com/sakibmuhtadee/MICT_2000_Project
 * branch            main       -> FETCH_HEAD


Already up to date.
[main 1a14ced] pushed forensic dataset
 1 file changed, 62292 insertions(+)
 create mode 100644 forensic_dataset_bn.csv
Successfully pushed forensic_dataset_bn.csv to MICT_2000_Project/main branch


To https://github.com/sakibmuhtadee/MICT_2000_Project.git
   353cef0..1a14ced  main -> main
